In [ ]:
# ============================================================
# AI Innovators Lab: Student Performance Prediction
# ============================================================
# This notebook teaches students how a simple AI model can use
# student information to make a prediction.
#
# Important idea:
# The model does NOT truly understand students like a teacher does.
# It only learns patterns from the data that we give it.
#
# In this project, the model predicts one of two classes:
# 1. Pass
# 2. Needs Improvement
#
# The inputs are simple student-related features:
# - studytime: how many hours the student studies each week
# - absences: how many school absences the student has
# - schoolsup: whether the student receives school support
# - famsup: whether the student receives family support
# - paid: whether the student attends extra paid classes
#
# Dataset:
# UCI Student Performance Dataset
# Link: https://archive.ics.uci.edu/dataset/320/student%2Bperformance
#
# Camp version:
# The original dataset has many columns. For this summer camp version,
# we use only a few beginner-friendly columns so students can understand
# the complete machine learning process.
# ============================================================

# Show a welcome message when the notebook starts.
print("Welcome to the Student Performance Prediction Project!")

# Tell students the main goal of this project.
print("Goal: Build an AI model that predicts student performance.")


In [ ]:
# ============================================================
# Step 1: Import required libraries
# ============================================================
# A library is a collection of pre-written code.
# Instead of writing everything ourselves, we import libraries
# that already know how to work with tables, graphs, and AI models.
# ============================================================

# Import pandas and give it the nickname pd.
# pandas helps us work with table-like data, similar to spreadsheets.
import pandas as pd

# Import numpy and give it the nickname np.
# numpy helps with numbers and arrays. We do not use it much here,
# but it is commonly used in AI projects.
import numpy as np

# Import matplotlib's plotting tool and give it the nickname plt.
# matplotlib helps us draw graphs and charts.
import matplotlib.pyplot as plt

# Import os.
# os helps Python check whether files exist on the computer or in Colab.
import os

# Import train_test_split.
# This helps us divide the dataset into training data and testing data.
from sklearn.model_selection import train_test_split

# Import the Decision Tree model.
# A decision tree is an AI model that makes predictions using yes/no style questions.
from sklearn.tree import DecisionTreeClassifier

# Import plot_tree.
# plot_tree lets us draw the decision tree so students can see how it thinks.
from sklearn.tree import plot_tree

# Import accuracy_score.
# accuracy_score tells us what percentage of predictions were correct.
from sklearn.metrics import accuracy_score

# Import classification_report.
# classification_report gives more details about model performance.
from sklearn.metrics import classification_report

# Import confusion_matrix.
# confusion_matrix shows correct and incorrect predictions in a table.
from sklearn.metrics import confusion_matrix

# Print a message so students know this cell ran successfully.
print("Libraries loaded successfully!")


In [ ]:
# ============================================================
# Step 2: Load the student performance dataset
# ============================================================
# The dataset is stored online as a CSV file.
# CSV means Comma-Separated Values, but this specific file uses semicolons.
# A CSV file is a simple table file that Python can read.
# ============================================================

# Import pandas again in this cell.
# This makes the cell easier to run by itself if needed.
import pandas as pd

# Import os again in this cell.
# This lets us check for a local backup file if the online file does not load.
import os

# Print a message so students know we are about to load data.
print("Loading dataset...")

# Store the web address of the dataset in a variable named url.
# A variable is like a labeled box that stores information.
url = "https://raw.githubusercontent.com/arunk13/MSDA-Assignments/master/IS607Fall2015/Assignment3/student-mat.csv"

# Start a try block.
# Python will first try to run the code inside this block.
try:

    # Read the online CSV file into a pandas table called data.
    # sep=";" tells pandas that columns are separated by semicolons.
    data = pd.read_csv(url, sep=";")

    # Print a success message if the online dataset loads correctly.
    print("Dataset loaded successfully from online CSV!")

# If something goes wrong in the try block, Python jumps to this except block.
except Exception as e:

    # Tell students that the online loading did not work.
    print("Online loading failed.")

    # Print the actual error message to help with debugging.
    print("Error:", e)

    # Check whether a local file named student-mat.csv exists.
    if os.path.exists("student-mat.csv"):

        # If the local file exists, load it instead of the online file.
        data = pd.read_csv("student-mat.csv", sep=";")

        # Print a success message for the local file.
        print("Dataset loaded successfully from local file!")

    # If neither the online file nor the local file works, use this else block.
    else:

        # Stop the program and explain what file is needed.
        raise FileNotFoundError(
            "Could not load the dataset online or locally. "
            "Please upload student-mat.csv to Colab."
        )

# Print the number of rows in the dataset.
# Each row represents one student.
print("Number of students:", len(data))

# Print the number of columns in the dataset.
# Each column represents one type of information about students.
print("Number of columns:", len(data.columns))

# Show the first five rows of the dataset.
# This helps students see what the data looks like.
print(data.head())


In [ ]:
# ============================================================
# Step 3: Create a smaller camp-friendly dataset
# ============================================================
# The original dataset has many columns, which can be confusing.
# Here, we keep only the columns that are easier to understand.
#
# Selected input features:
# - studytime: weekly study time category
# - absences: number of school absences
# - schoolsup: extra educational support from school
# - famsup: educational support from family
# - paid: extra paid classes
#
# Original grade column:
# - G3: final grade from 0 to 20
#
# New target column:
# - performance: Pass or Needs Improvement
#
# Rule for this camp activity:
# - If G3 is 10 or higher, the student is labeled Pass.
# - If G3 is below 10, the student is labeled Needs Improvement.
# ============================================================

# Select only the columns we want to use for the beginner version.
# The double brackets create a smaller table with these specific columns.
simple_data = data[["studytime", "absences", "schoolsup", "famsup", "paid", "G3"]].copy()

# Create a new column named performance.
# apply() runs a small function on every grade in the G3 column.
simple_data["performance"] = simple_data["G3"].apply(

    # This lambda function checks each grade.
    # If grade >= 10, it returns Pass. Otherwise, it returns Needs Improvement.
    lambda grade: "Pass" if grade >= 10 else "Needs Improvement"
)

# Print a message so students know the smaller dataset is ready.
print("Simple dataset created!")

# Show the first five rows of the smaller dataset.
print(simple_data.head())


In [ ]:
# ============================================================
# Step 4: Explain the selected features
# ============================================================
# This cell prints a plain-English explanation of each column.
# Students should understand what the data means before training the model.
# ============================================================

# Store a long explanation inside a text variable.
# Triple quotes allow us to write text across many lines.
feature_explanation = """
Feature Meaning:

studytime:
1 = less than 2 hours/week
2 = 2 to 5 hours/week
3 = 5 to 10 hours/week
4 = more than 10 hours/week

absences:
Number of school absences

schoolsup:
Extra educational support from school
yes = student receives support
no  = student does not receive support

famsup:
Family educational support
yes = student receives family support
no  = student does not receive family support

paid:
Extra paid classes
yes = student attends extra paid classes
no  = student does not attend extra paid classes

G3:
Final grade from 0 to 20

performance:
Pass or Needs Improvement
"""

# Print the explanation so students can read it in the notebook output.
print(feature_explanation)


In [ ]:
# ============================================================
# Step 5: Visualize Pass vs Needs Improvement
# ============================================================
# A graph helps us understand the dataset before training the AI model.
# Here, we count how many students are in each performance category.
# ============================================================

# Print a blank line first, then print the title Performance counts.
print("\nPerformance counts:")

# Count how many students are Pass and how many are Needs Improvement.
print(simple_data["performance"].value_counts())

# Create a new graph with a width of 6 inches and a height of 4 inches.
plt.figure(figsize=(6, 4))

# Count each category and draw the result as a bar chart.
simple_data["performance"].value_counts().plot(kind="bar")

# Add a title to the graph.
plt.title("Student Performance Categories")

# Label the x-axis.
plt.xlabel("Category")

# Label the y-axis.
plt.ylabel("Number of Students")

# Keep the category labels horizontal so they are easier to read.
plt.xticks(rotation=0)

# Display the graph.
plt.show()


In [ ]:
# ============================================================
# Step 6: Convert words into numbers
# ============================================================
# Machine learning models work with numbers, not words.
# This means we must convert text values like yes, no, Pass,
# and Needs Improvement into numbers.
#
# We convert:
# yes -> 1
# no  -> 0
#
# We also convert:
# Pass -> 1
# Needs Improvement -> 0
# ============================================================

# Store the names of columns that contain yes/no answers.
yes_no_columns = ["schoolsup", "famsup", "paid"]

# Loop through each column name in the yes_no_columns list.
# A loop repeats the same action for multiple items.
for col in yes_no_columns:

    # Replace yes with 1 and no with 0 in the current column.
    # map() changes values according to the dictionary we provide.
    simple_data[col] = simple_data[col].map({

        # Change the word yes to the number 1.
        "yes": 1,

        # Change the word no to the number 0.
        "no": 0
    })

# Create a new numeric target column called performance_number.
# This is the version of the answer that the AI model can learn from.
simple_data["performance_number"] = simple_data["performance"].map({

    # Change Needs Improvement to 0.
    "Needs Improvement": 0,

    # Change Pass to 1.
    "Pass": 1
})

# Print a message so students know the conversion is complete.
print("Text converted to numbers!")

# Show the first five rows after conversion.
print(simple_data.head())


In [ ]:
# ============================================================
# Step 7: Select input features and target label
# ============================================================
# In machine learning, we usually separate the data into X and y.
#
# X = the input information we give to the AI model.
# y = the correct answer we want the AI model to learn to predict.
#
# In this project:
# X contains studytime, absences, schoolsup, famsup, and paid.
# y contains performance_number, where 1 means Pass and 0 means Needs Improvement.
# ============================================================

# Create X using the columns the model will use as input.
X = simple_data[["studytime", "absences", "schoolsup", "famsup", "paid"]]

# Create y using the column the model should predict.
y = simple_data["performance_number"]

# Print a heading for the input features.
print("Input features:")

# Show the first five rows of X.
print(X.head())

# Print a heading for the target labels.
print("\nTarget labels:")

# Show the first five values of y.
print(y.head())


In [ ]:
# ============================================================
# Step 8: Split data into training and testing sets
# ============================================================
# We do not want to test the model on the same exact data it studied.
# That would be like giving students the answers before an exam.
#
# Training data teaches the AI model.
# Testing data checks whether the AI model learned useful patterns.
# ============================================================

# Split X and y into training and testing parts.
X_train, X_test, y_train, y_test = train_test_split(

    # X contains the input columns.
    X,

    # y contains the correct answers.
    y,

    # Use 25% of the data for testing.
    test_size=0.25,

    # random_state makes the split repeatable.
    # This means students get the same result each time they run it.
    random_state=42,

    # stratify=y keeps the Pass and Needs Improvement ratio similar
    # in both the training set and the testing set.
    stratify=y
)

# Print how many examples are used for training.
print("Training examples:", len(X_train))

# Print how many examples are used for testing.
print("Testing examples:", len(X_test))


In [ ]:
# ============================================================
# Step 9: Build and train the AI model
# ============================================================
# We use a Decision Tree Classifier.
#
# A decision tree works like a flowchart.
# It asks questions such as:
# - Is absences less than or equal to a certain number?
# - Is studytime greater than a certain value?
# Based on the answers, it moves down the tree and makes a prediction.
# ============================================================

# Create the decision tree model.
model = DecisionTreeClassifier(

    # max_depth controls how many levels the tree can grow.
    # A smaller depth makes the tree easier for students to understand.
    max_depth=4,

    # random_state makes the model behavior repeatable.
    random_state=42
)

# Train the model using the training inputs and training answers.
# This is the step where the model learns patterns from the data.
model.fit(X_train, y_train)

# Print a message when training is finished.
print("Model training completed!")


In [ ]:
# ============================================================
# Step 10: Evaluate the model
# ============================================================
# Evaluation means checking how well the model performs.
# We use the testing data because the model did not train on it.
# ============================================================

# Ask the trained model to predict answers for the testing inputs.
y_pred = model.predict(X_test)

# Compare the real answers with the predicted answers.
# accuracy tells us the percentage of correct predictions.
accuracy = accuracy_score(y_test, y_pred)

# Print the accuracy as a percentage.
print("Model Accuracy:", round(accuracy * 100, 2), "%")

# Print a heading for the classification report.
print("\nClassification Report:")

# Print precision, recall, and F1-score for both classes.
# These are more detailed ways to measure model performance.
print(classification_report(

    # These are the real answers from the testing set.
    y_test,

    # These are the model's predicted answers.
    y_pred,

    # These names make the report easier to read.
    target_names=["Needs Improvement", "Pass"]
))

# Print a heading for the confusion matrix.
print("\nConfusion Matrix:")

# Print the confusion matrix as numbers.
# It shows how many predictions were correct and incorrect.
print(confusion_matrix(y_test, y_pred))


In [ ]:
# ============================================================
# Step 11: Visualize the confusion matrix
# ============================================================
# A confusion matrix helps us see the model's mistakes.
# Rows are the true labels.
# Columns are the predicted labels.
# ============================================================

# Create the confusion matrix using true answers and predicted answers.
cm = confusion_matrix(y_test, y_pred)

# Create a new graph with a width of 5 inches and a height of 4 inches.
plt.figure(figsize=(5, 4))

# Show the confusion matrix as an image.
plt.imshow(cm)

# Add a title to the graph.
plt.title("Confusion Matrix")

# Add a color bar so students can see how large each number is.
plt.colorbar()

# Label the x-axis positions with prediction names.
plt.xticks([0, 1], ["Needs Improvement", "Pass"])

# Label the y-axis positions with true label names.
plt.yticks([0, 1], ["Needs Improvement", "Pass"])

# Label the x-axis.
plt.xlabel("Predicted Label")

# Label the y-axis.
plt.ylabel("True Label")

# Loop through the two rows of the matrix.
for i in range(2):

    # Loop through the two columns of the matrix.
    for j in range(2):

        # Write the number from the confusion matrix inside the correct square.
        plt.text(j, i, cm[i, j], ha="center", va="center")

# Display the confusion matrix graph.
plt.show()


In [ ]:
# ============================================================
# Step 12: Visualize the decision tree
# ============================================================
# This picture helps students see how the model makes decisions.
# Each box in the tree is a question or a decision.
# ============================================================

# Create a large figure so the tree is easier to read.
plt.figure(figsize=(18, 8))

# Draw the trained decision tree model.
plot_tree(

    # This is the model we trained earlier.
    model,

    # Show the names of the input features in the tree.
    feature_names=X.columns,

    # Show class names instead of only 0 and 1.
    class_names=["Needs Improvement", "Pass"],

    # Fill boxes with color so different classes are easier to see.
    filled=True,

    # Round the corners of the boxes for a cleaner look.
    rounded=True,

    # Set the font size inside the tree boxes.
    fontsize=9
)

# Add a title above the tree.
plt.title("Decision Tree for Student Performance Prediction")

# Display the decision tree.
plt.show()


In [ ]:
# ============================================================
# Step 13: Test the model with custom student examples
# ============================================================
# Now we create our own fake student examples.
# This lets students see how different inputs can change the prediction.
#
# studytime:
# 1 = less than 2 hours/week
# 2 = 2 to 5 hours/week
# 3 = 5 to 10 hours/week
# 4 = more than 10 hours/week
#
# schoolsup, famsup, paid:
# 1 = yes
# 0 = no
# ============================================================

# Create a small table with three custom student examples.
custom_students = pd.DataFrame([

    # First example: low study time, many absences, and no extra support.
    {
        "studytime": 1,
        "absences": 12,
        "schoolsup": 0,
        "famsup": 0,
        "paid": 0
    },

    # Second example: higher study time, few absences, and several supports.
    {
        "studytime": 3,
        "absences": 2,
        "schoolsup": 1,
        "famsup": 1,
        "paid": 1
    },

    # Third example: medium study time, some absences, and family support.
    {
        "studytime": 2,
        "absences": 5,
        "schoolsup": 0,
        "famsup": 1,
        "paid": 0
    }
])

# Ask the model to predict Pass or Needs Improvement for each custom student.
custom_predictions = model.predict(custom_students)

# Ask the model for prediction probabilities.
# Probabilities show how confident the model is for each class.
custom_probabilities = model.predict_proba(custom_students)

# Print a heading before showing the custom predictions.
print("Custom Student Predictions:\n")

# Loop through each custom student example.
for i in range(len(custom_students)):

    # Get the prediction for the current student.
    prediction = custom_predictions[i]

    # Get the highest probability and convert it to a percentage.
    confidence = max(custom_probabilities[i]) * 100

    # Convert the numeric prediction into a readable label.
    label = "Pass" if prediction == 1 else "Needs Improvement"

    # Print the example number.
    print("Student Example", i + 1)

    # Print the input values for this student.
    print(custom_students.iloc[i])

    # Print the model's predicted label.
    print("Prediction:", label)

    # Print the model's confidence score.
    print("Confidence:", round(confidence, 2), "%")

    # Print a line to separate this example from the next one.
    print("-" * 60)


In [ ]:
# ============================================================
# Step 14: Let students enter their own values
# ============================================================
# This cell turns the model into a small interactive demo.
# Students type values, and the model predicts the performance category.
# ============================================================

# Print an instruction message for students.
print("Now test your own student example.")

# Explain the studytime choices.
print("For studytime, choose:")

# Explain option 1.
print("1 = less than 2 hours/week")

# Explain option 2.
print("2 = 2 to 5 hours/week")

# Explain option 3.
print("3 = 5 to 10 hours/week")

# Explain option 4.
print("4 = more than 10 hours/week")

# Ask the student to enter studytime.
# input() receives text from the keyboard.
# int() changes the typed text into an integer number.
student_studytime = int(input("Enter studytime (1, 2, 3, or 4): "))

# Ask the student to enter the number of absences.
student_absences = int(input("Enter number of absences: "))

# Ask whether the student receives extra school support.
# lower() changes the answer to lowercase so Yes, YES, and yes are treated similarly.
student_schoolsup = input("Extra school support? (yes/no): ").lower()

# Ask whether the student receives family support.
student_famsup = input("Family support? (yes/no): ").lower()

# Ask whether the student attends extra paid classes.
student_paid = input("Extra paid classes? (yes/no): ").lower()

# Create a one-row table for the student-entered example.
student_example = pd.DataFrame([{

    # Store the student's studytime choice.
    "studytime": student_studytime,

    # Store the student's number of absences.
    "absences": student_absences,

    # Convert yes to 1. Any other answer becomes 0.
    "schoolsup": 1 if student_schoolsup == "yes" else 0,

    # Convert yes to 1. Any other answer becomes 0.
    "famsup": 1 if student_famsup == "yes" else 0,

    # Convert yes to 1. Any other answer becomes 0.
    "paid": 1 if student_paid == "yes" else 0
}])

# Ask the model to predict the class for the student-entered example.
# [0] gets the first and only prediction from the result.
student_prediction = model.predict(student_example)[0]

# Ask the model for the probability of each class for this example.
student_probability = model.predict_proba(student_example)[0]

# Check whether the model predicted class 1.
if student_prediction == 1:

    # If the prediction is 1, print Pass.
    print("\nThe AI predicts: Pass")

# If the prediction is not 1, run this else block.
else:

    # If the prediction is 0, print Needs Improvement.
    print("\nThe AI predicts: Needs Improvement")

# Print the model's confidence as a percentage.
print("Confidence:", round(max(student_probability) * 100, 2), "%")


In [ ]:
# ============================================================
# Step 15: Feature importance
# ============================================================
# Feature importance tells us which input columns helped the
# decision tree the most when making predictions.
#
# Important note:
# Feature importance does NOT prove that one thing causes another.
# It only tells us what the model used most in this dataset.
# ============================================================

# Create a new table that stores each feature name and its importance score.
feature_importance = pd.DataFrame({

    # Store the names of the input columns.
    "Feature": X.columns,

    # Store the importance scores learned by the decision tree.
    "Importance": model.feature_importances_
})

# Sort the table so the most important feature appears first.
feature_importance = feature_importance.sort_values(

    # Sort using the Importance column.
    by="Importance",

    # Use descending order so larger values come first.
    ascending=False
)

# Print a heading.
print("Feature Importance:")

# Print the feature importance table.
print(feature_importance)

# Create a new graph with a width of 8 inches and a height of 5 inches.
plt.figure(figsize=(8, 5))

# Draw a horizontal bar chart of feature importance values.
plt.barh(feature_importance["Feature"], feature_importance["Importance"])

# Add a title to the chart.
plt.title("Which Features Helped the Model Most?")

# Label the x-axis.
plt.xlabel("Importance")

# Label the y-axis.
plt.ylabel("Feature")

# Put the most important feature at the top of the chart.
plt.gca().invert_yaxis()

# Display the chart.
plt.show()


In [ ]:
# ============================================================
# Step 16: Save the model
# ============================================================
# Saving the model means we can use it later without retraining it.
# The pickle library can save Python objects, including trained models.
# ============================================================

# Import pickle, which helps save Python objects to files.
import pickle

# Open a new file in write-binary mode.
# "wb" means write binary, which is needed for pickle files.
with open("student_performance_model.pkl", "wb") as file:

    # Save the trained model inside the file.
    pickle.dump(model, file)

# Print a message so students know the model was saved.
print("Model saved as student_performance_model.pkl")


In [ ]:
# ============================================================
# Step 17: Reflection questions for final showcase
# ============================================================
# These questions help students explain not only what they built,
# but also how the model works and how to use AI responsibly.
# ============================================================

# Store all reflection questions in a Python list.
reflection_questions = [

    # Question about the project goal.
    "1. What problem does your AI model solve?",

    # Question about the dataset.
    "2. What dataset did you use?",

    # Question about the input features.
    "3. What inputs did your model use?",

    # Question about the target label.
    "4. What does Pass mean in this project?",

    # Question about model performance.
    "5. What was your model accuracy?",

    # Question about feature importance.
    "6. Which feature seemed most important?",

    # Question about model mistakes.
    "7. Did the model make any mistakes?",

    # Question about responsible AI use.
    "8. How could this AI model be used responsibly?",

    # Question about fairness and human decision making.
    "9. Why should humans be careful when using AI to make decisions about students?"
]

# Print a heading before the questions.
print("\nReflection Questions:")

# Loop through each question in the list.
for question in reflection_questions:

    # Print the current question.
    print(question)


In [ ]:
# ============================================================
# Step 18: Presentation template for students
# ============================================================
# Students can use this template to prepare their final showcase.
# It helps them explain the project in a clear and responsible way.
# ============================================================

# Store the presentation template as a long multi-line string.
presentation_template = """
Project Title:
Student Performance Prediction AI Model

Team Members:
[Names]

Problem:
We wanted to build an AI model that predicts whether a student may pass or need improvement.

Dataset:
We used the UCI Student Performance Dataset.
Dataset Link:
https://archive.ics.uci.edu/dataset/320/student%2Bperformance

Inputs:
Study time, absences, school support, family support, and extra classes.

Prediction:
Pass or Needs Improvement

Model Used:
Decision Tree Classifier

Result:
Our model reached approximately ____% accuracy.

Demo:
We entered a new student example, and the model predicted whether the student may pass or need improvement.

What We Learned:
We learned how AI can use data to make predictions, but also why humans must use AI responsibly.

Responsible AI Note:
This model should not be used to judge real students unfairly. It should only support learning and discussion.

Future Improvement:
We could include more useful features, collect better data, or use the model only as an early-warning support tool.
"""

# Print the presentation template so students can copy and edit it.
print(presentation_template)
